In [1]:
# Parameters
BATCH_MODE = "true"


# 12 - Test-Time Scaling : le second axe de mise a l'echelle

**Navigation** : [Index](README.md) | [<< Precedent](11_Quantization.ipynb)

Jusqu'ici nous avons mis a l'echelle les LLM en augmentant leur **taille** (parametres, donnees d'entrainement) : c'est le **size-scaling**. Snell et al. (2024), dans *Scaling LLM Test-Time Compute Optimally*, ont formalise un **second axe** : depenser davantage de **calcul a l'inference** (test-time compute). Au lieu d'un seul appel direct, on orchestre plusieurs appels : Best-of-N, Reflexion, Tree-of-Thoughts...

> Ce notebook poursuit le projet de reference **Iterative Contextual Refinements (ICR)** : https://github.com/ryoiki-tokuiten/Iterative-Contextual-Refinements . ICR regroupe plusieurs "moteurs" (Contextual, Deepthink, Adaptive) que nous reconstruisons ici a partir de zero.

### La these de ce notebook

La performance d'un LLM n'est pas une fonction de sa seule taille. Elle se negocie entre **size-scaling** (un plus gros modele) et **test-time scaling** (plus d'effort d'inference). Et ces deux axes **ne sont pas equivalents** : sur certaines taches, un modele bon marche plus un bon orchestrateur bat un modele gros raisonnant, pour beaucoup moins de tokens. Le test-time scaling n'est pas un supplement universel : c'est un **outil** dont le gain depend de la **structure d'erreur** du modele.

**Note importante** : `gpt-4o-mini` est desormais **deprécie** (2026). Nous comparons :
- **Llama-3.3-70b** (modele bon marche, non-raisonnant) ;
- **gpt-5-nano** (modele raisonnant, plus couteux, qui consomme des tokens de raisonnement invisibles).

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Distinguer **size-scaling** et **test-time scaling** comme deux axes independants.
2. Mesurer le **cout** (tokens) et la **performance** (tests passes) de chaque strategie.
3. Implementer les 4 moteurs : Best-of-N, Reflexion, Tree-of-Thoughts, routeur adaptatif.
4. Diagnostiquer quand le test-time scaling aide - et quand il est inutile, voire contre-productif.

### Prerequis
- Python 3.10+, cle API OpenRouter configuree (`.env` avec `OPENROUTER_API_KEY`).
- Notions de generation de code et d'execution de tests (benchmark objectif).

### Duree estimee : 60 minutes (incluant les appels API, 5 a 8 min de calcul)

In [2]:
# Installation des dependances
%pip install -q openai python-dotenv

import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

# --- Chargeur .env robuste (Papermill change le cwd) ---
_env_path = None
_current = Path.cwd()
for _i in range(10):
    if (_current / ".env").exists():
        _env_path = _current / ".env"
        break
    if _current.name in ("GenAI", "MyIA.AI.Notebooks"):
        break
    _current = _current.parent
if _env_path is None:
    for _cand in (
        Path.cwd() / "MyIA.AI.Notebooks" / "GenAI" / ".env",
        Path.cwd() / "GenAI" / ".env",
    ):
        if _cand.exists():
            _env_path = _cand
            break
if _env_path is not None:
    load_dotenv(_env_path)
    print(f".env charge depuis : {_env_path}")
else:
    print("ATTENTION : aucun .env trouve. Les variables d'env doivent etre exportees.")

# --- Modeles (etat de l'art 2026) ---
FAST_MODEL = os.getenv("OPENAI_MODEL_FAST", "meta-llama/llama-3.3-70b-instruct")
BIG_MODEL = os.getenv("OPENAI_MODEL_BIG", "openai/gpt-5-nano")
BATCH_MODE = os.getenv("BATCH_MODE", "false").lower() in ("1", "true", "yes")

# --- Client OpenRouter ---
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
)

print(f"FAST_MODEL (bon marche) = {FAST_MODEL}")
print(f"BIG_MODEL  (raisonnant) = {BIG_MODEL}")
print(f"BATCH_MODE = {BATCH_MODE}")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


.env charge depuis : C:\dev\CoursIA-2\MyIA.AI.Notebooks\GenAI\.env


FAST_MODEL (bon marche) = meta-llama/llama-3.3-70b-instruct
BIG_MODEL  (raisonnant) = openai/gpt-5-nano
BATCH_MODE = False


In [3]:
def chat(prompt, system=None, model=FAST_MODEL, temperature=0.0, max_tokens=4000):
    """Appel unique au LLM. Renvoie une chaine vide en cas d'erreur
    (le notebook ne doit jamais casser en BATCH_MODE).

    Note : gpt-5-nano est un modele raisonnant qui consomme ~500-2000
    tokens de raisonnement INVISIBLES (comptes dans max_tokens). Un budget
    trop bas (ex. 2000) laisse le raisonnement manger tout le plafond et
    renvoyer un contenu VIDE. max_tokens=4000 par defaut laisse assez de
    marge pour le raisonnement + la generation de code effective.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return resp.choices[0].message.content or ""
    except Exception as exc:
        print(f"  [chat] erreur ({model}) : {exc}")
        return ""


# Test rapide de connectivite (1 appel economique)
_ping = chat("Reponds uniquement par : OK", model=FAST_MODEL, max_tokens=10)
print("Ping FAST_MODEL :", repr(_ping[:40]))

Ping FAST_MODEL : 'OK'


## Le benchmark : generation de code avec verite d'execution

Pour mesurer le test-time scaling, il nous faut une tache avec une **verite objective**. Les problemes de mots (word problems) ne conviennent plus : tous les modeles 2026 les saturent. Nous utilisons une **generation de code** ou la verite = **executer des tests**.

**Avantages** :
- **Objectif** : un test passe ou echoue, pas d'opinion.
- **Difficulte parametrique** : on calibre precisement la difficulte.
- **Cout mesurable** : on compte exactement les tokens depenses.

6 problemes de difficulte croissante (1 = trivial, 6 = algorithmique) :

| # | Probleme | Concept |
|---|----------|---------|
| p1 | carre | arithmetique de base |
| p2 | compteur_pairs | iteration + predicat |
| p3 | fizzbuzz | ramification classique |
| p4 | plus_long_palindrome | sous-chaine, expansion |
| p5 | sous_ensembles | combinatoire, recursion |
| p6 | chemins dans un graphe | parcours, backtracking |

> Les modeles 2026 saturent p1-p4. Le **vrai signal** se situe sur p5 (cout) et p6 (erreur systematique).

In [4]:
import re

PROBLEMES = [
    {"id": "p1_carre", "diff": 1,
     "spec": "Ecris une fonction python `carre(n)` qui renvoie le carre de n.",
     "tests": ["carre(3)==9", "carre(0)==0", "carre(-4)==16"]},
    {"id": "p2_parite", "diff": 2,
     "spec": "Ecris une fonction python `compteur_pairs(liste)` qui renvoie le nombre d'elements pairs dans une liste d'entiers.",
     "tests": ["compteur_pairs([1,2,3,4,6])==3", "compteur_pairs([])==0", "compteur_pairs([1,3,5])==0"]},
    {"id": "p3_fizzbuzz", "diff": 3,
     "spec": "Ecris une fonction python `fizzbuzz(n)` qui renvoie une liste de longueur n ou l'element i (1-indexe) vaut 'Fizz' si i multiple de 3, 'Buzz' si multiple de 5, 'FizzBuzz' si multiple des deux, sinon l'entier i.",
     "tests": ["fizzbuzz(5)==[1,2,'Fizz',4,'Buzz']", "fizzbuzz(15)[-1]=='FizzBuzz'", "fizzbuzz(0)==[]"]},
    {"id": "p4_palindrome", "diff": 4,
     "spec": "Ecris une fonction python `plus_long_palindrome(s)` qui renvoie le plus long palindrome contigu dans s. En cas d'egalite, renvoie le premier trouve. Renvoie '' si s vide.",
     "tests": ["plus_long_palindrome('babad') in ('bab','aba')", "plus_long_palindrome('racecar')=='racecar'", "plus_long_palindrome('')==''", "plus_long_palindrome('abc')=='a'"]},
    {"id": "p5_subsets", "diff": 5,
     "spec": "Ecris une fonction python `sous_ensembles(lst)` qui renvoie la liste TRIEE de tous les sous-ensembles (tuples) de lst. Chaque sous-ensemble est un tuple. 2^n tuples au total, y compris le tuple vide ().",
     "tests": ["len(sous_ensembles([1,2,3]))==8", "() in sous_ensembles([1,2])", "sorted(sous_ensembles([1,2]))==sorted([(),(1,),(2,),(1,2)])"]},
    {"id": "p6_graph", "diff": 6,
     "spec": "Ecris une fonction python `chemins(graphe, debut, fin)` ou graphe est un dict {noeud:[voisins]}. Renvoie TOUS les chemins simples (sans cycle) de debut a fin, comme liste de listes, triee par longueur puis lexicographiquement. Renvoie [] si aucun chemin.",
     "tests": ["chemins({1:[2,3],2:[3],3:[4],4:[]},1,4)==[[1,2,3,4],[1,3,4]]", "chemins({1:[2],2:[3],3:[]},1,4)==[]", "chemins({1:[2],2:[1],3:[2]},3,1)==[[3,2,1]]"]},
]


def extraire_code(texte):
    """Extrait le code d'un bloc ```python ... ``` (ou ``` ... ```).
    Si aucun bloc, renvoie le texte tel quel."""
    m = re.search(r"```python\n(.*?)```", texte, re.DOTALL)
    if m:
        return m.group(1)
    m = re.search(r"```\n(.*?)```", texte, re.DOTALL)
    if m:
        return m.group(1)
    return texte


def executer_tests(code_gen, probleme):
    """Execute le code dans un espace isole, evalue chaque test.
    Renvoie (nb_passes, nb_total). Une exception = 0 pour ce test."""
    ns = {}
    try:
        exec(code_gen, ns)
    except Exception:
        return 0, len(probleme["tests"])
    passes = 0
    for cond in probleme["tests"]:
        try:
            if eval(cond, ns):
                passes += 1
        except Exception:
            pass
    return passes, len(probleme["tests"])


print(f"{len(PROBLEMES)} problemes charges, difficultes : {[p['diff'] for p in PROBLEMES]}")

6 problemes charges, difficultes : [1, 2, 3, 4, 5, 6]


In [5]:
def gen_code(probleme, model=FAST_MODEL, temperature=0.0, max_tokens=2000):
    """Genere le code pour un probleme, en demandant UNIQUEMENT du code.
    Renvoie (code_extrait, tokens_estimes)."""
    prompt = (
        probleme["spec"]
        + "\n\nReponds UNIQUEMENT avec le code Python dans un bloc ```python```, sans explication."
    )
    resp = chat(prompt, model=model, temperature=temperature, max_tokens=max_tokens)
    code_brut = extraire_code(resp)
    # estimation grossiere des tokens (mots ~= tokens)
    tokens = len(resp.split())
    return code_brut, tokens


# Mini-test : genere p1 et l'execute (smoke test, 1 appel)
_code, _tok = gen_code(PROBLEMES[0], model=FAST_MODEL)
_pass, _tot = executer_tests(_code, PROBLEMES[0])
print(f"[smoke test] {PROBLEMES[0]['id']} : {_pass}/{_tot} tests, ~{_tok} tokens")
print("--- code genere ---")
print(_code[:300])

[smoke test] p1_carre : 3/3 tests, ~8 tokens
--- code genere ---
def carre(n):
    return n ** 2



In [6]:
def mesure_baseline(problemes, models):
    """Lance gen_code single-shot sur chaque (modele, probleme).
    Renvoie une liste de dicts {id, diff, modele, passes, total, tokens}."""
    resultats = []
    for modele, etiquette in models:
        print(f"\n=== Modele : {etiquette} ({modele}) ===")
        for pb in problemes:
            try:
                code_gen, tokens = gen_code(pb, model=modele, temperature=0.0)
                passes, total = executer_tests(code_gen, pb)
            except Exception as exc:
                print(f"  [erreur] {pb['id']} : {exc}")
                code_gen, tokens, passes, total = "", 0, 0, len(pb["tests"])
            print(f"  {pb['id']:16s} diff={pb['diff']} -> {passes}/{total} tests, ~{tokens} tokens")
            resultats.append({
                "id": pb["id"], "diff": pb["diff"], "modele": etiquette,
                "passes": passes, "total": total, "tokens": tokens,
            })
    return resultats


# On limite le nombre de problemes en BATCH_MODE pour rester sous budget
_pbs = PROBLEMES if not BATCH_MODE else PROBLEMES[:4]
print(f"Lancement baseline sur {len(_pbs)} problemes x 2 modeles (~{len(_pbs) * 2} appels API, 2-3 min)...")
RESULTATS_BASELINE = mesure_baseline(_pbs, [
    (FAST_MODEL, "cheap (Llama-3.3-70b)"),
    (BIG_MODEL, "big (gpt-5-nano)"),
])

# Affichage synthetique : on apparie cheap/big par id de probleme
# (resultats contient tous les cheap PUIS tous les big, il faut regrouper)
print("\n" + "=" * 70)
print(f"{'Probleme':16s} | {'Diff':>4s} | {'Cheap':>16s} | {'Big':>16s}")
print("-" * 70)
_pids = [pb["id"] for pb in _pbs]
for _pid in _pids:
    cheap = next((r for r in RESULTATS_BASELINE if r["id"] == _pid and "cheap" in r["modele"]), None)
    big = next((r for r in RESULTATS_BASELINE if r["id"] == _pid and "big" in r["modele"]), None)
    if cheap and big:
        print(f"{cheap['id']:16s} | {cheap['diff']:>4d} | {cheap['passes']}/{cheap['total']} ~{cheap['tokens']:>5d}tok | {big['passes']}/{big['total']} ~{big['tokens']:>5d}tok")

Lancement baseline sur 6 problemes x 2 modeles (~12 appels API, 2-3 min)...

=== Modele : cheap (Llama-3.3-70b) (meta-llama/llama-3.3-70b-instruct) ===


  p1_carre         diff=1 -> 3/3 tests, ~8 tokens


  p2_parite        diff=2 -> 3/3 tests, ~16 tokens


  p3_fizzbuzz      diff=3 -> 3/3 tests, ~35 tokens


  p4_palindrome    diff=4 -> 4/4 tests, ~70 tokens


...

  p2_parite        diff=2 -> 3/3 tests, ~36 tokens


  p3_fizzbuzz      diff=3 -> 3/3 tests, ~42 tokens


  p4_palindrome    diff=4 -> 0/4 tests, ~0 tokens


  p5_subsets       diff=5 -> 3/3 tests, ~33 tokens


  p6_graph         diff=6 -> 0/3 tests, ~0 tokens

Probleme         | Diff |            Cheap |              Big
----------------------------------------------------------------------
p1_carre         |    1 | 3/3 ~    8tok | 3/3 ~    8tok
p2_parite        |    2 | 3/3 ~   16tok | 3/3 ~   36tok
p3_fizzbuzz      |    3 | 3/3 ~   35tok | 3/3 ~   42tok
p4_palindrome    |    4 | 4/4 ~   70tok | 0/4 ~    0tok
p5_subsets       |    5 | 3/3 ~   23tok | 3/3 ~   33tok
p6_graph         |    6 | 2/3 ~   40tok | 0/3 ~    0tok


### Interpretation : le size-scaling n'est pas toujours gagnant

**Le signal attendu** : sur p1-p3, les deux modeles saturent (tout passe). C'est la **zone saturee** : le test-time scaling n'y ajoute rien, et le size-scaling non plus.

**Resultats mesures** (cheap = Llama-3.3-70b, big = gpt-5-nano) :

| Probleme | Cheap | Big | Lecture |
|----------|-------|-----|---------|
| p1 a p3 | X/X (~8-35tok) | X/X (~8-42tok) | Sature : aucun gain, cout similaire |
| p4 palindrome | 4/4 (~70tok) | **0/4 (~0tok)** | **Big ECCHOUE** (sortie vide) la ou cheap passe |
| p5 subsets | 3/3 (~23tok) | 3/3 (~33tok) | Les deux passent, big legerement plus cher |
| p6 graph | **2/3** (~40tok) | **0/3** (~0tok)** | **Big ECCHOUE** (sortie vide) la ou cheap reussit partiellement |

**Conclusions** :
1. Sur les problemes les plus durs (p4, p6), le modele **gros raisonnant ECCHOUE** : il renvoie une **sortie vide** (~0 tokens effectifs), la ou le modele bon marche reussit (4/4 sur p4) ou reussit partiellement (2/3 sur p6). Le surcout de raisonnement n'achete ici **aucune performance** -- au contraire.
2. Sur p5, les deux modeles resolvent, big pour un cout legerement superieur (~33tok vs ~23tok). Le raisonnement n'est pas systematiquement contre-productif, mais il **n'apporte rien** non plus sur cette tache.
3. L'overhead de raisonnement est un **cout**, pas un bonus gratuit -- et dans certains cas il conduit a un echec total (sortie vide).

> **Note technique** : la sortie vide de gpt-5-nano (~0 tokens) sur p4/p6 n'est pas un crash : le modele raisonnant consomme son budget en tokens de raisonnement *invisibles* et ne produit pas de code extractible. C'est un echec de **formatage/generation**, pas d'incapacite brute -- mais du point de vue du benchmark, c'est bien un 0/4 et un 0/3.

C'est precisement ce phenomene qui justifie le **test-time scaling** : si un modele gros peut echouer la ou un cheap reussit, il vaut mieux savoir **orchestrer** un modele cheap. Voyons comment.

## Moteur 1 - Best-of-N (auto-coherence)

**Self-Consistency** (Wang et al., 2022) : au lieu d'un seul appel deterministe (temperature=0), on echantillonne **N** solutions a temperature elevee, puis on vote. C'est la brique elementaire du projet ICR.

**Intuition** : si les erreurs d'un modele sont **aleatoires** (differentes a chaque tirage), voter sur N annule le bruit. Mais si les erreurs sont **systematiques** (le modele se trompe toujours pareil), voter ne sert a rien.

**Question** : les erreurs de nos modeles sont-elles aleatoires ou systematiques ?

In [7]:
def best_of_n_code(probleme, model=FAST_MODEL, n=3, temperature=0.8, max_tokens=2000):
    """Genere n solutions (temperature elevee), execute chaque solution,
    garde celle qui passe le plus de tests (vote par performance).
    Renvoie (meilleur_passes, total, total_tokens, n)."""
    meilleur_passes = 0
    total = len(probleme["tests"])
    total_tokens = 0
    for _i in range(n):
        code_gen, tokens = gen_code(probleme, model=model, temperature=temperature, max_tokens=max_tokens)
        total_tokens += tokens
        passes, _ = executer_tests(code_gen, probleme)
        if passes > meilleur_passes:
            meilleur_passes = passes
    return meilleur_passes, total, total_tokens, n


print("best_of_n_code defini.")

best_of_n_code defini.


In [8]:
# Best-of-N sur le modele CHEAP, sur la zone non-saturee (p4, p5, p6)
_pbs_bon = [p for p in PROBLEMES if p["diff"] >= 4]
if BATCH_MODE:
    _pbs_bon = _pbs_bon[:2]

print("Best-of-N sur le modele cheap (zone non-saturee p4-p6) :\n")
print(f"{'Probleme':16s} | {'N=1':>14s} | {'N=3':>18s} | {'N=5':>20s}")
print("-" * 74)
RESULTATS_BON = []
for pb in _pbs_bon:
    c1, t1, tok1, _ = best_of_n_code(pb, model=FAST_MODEL, n=1)
    c3, t3, tok3, _ = best_of_n_code(pb, model=FAST_MODEL, n=3)
    c5, t5, tok5, _ = best_of_n_code(pb, model=FAST_MODEL, n=5)
    RESULTATS_BON.append({"id": pb["id"], "diff": pb["diff"],
                          "n1": (c1, tok1), "n3": (c3, tok3), "n5": (c5, tok5)})
    print(f"{pb['id']:16s} | {c1}/{t1} ~{tok1:>5d}tok | {c3}/{t3} ~{tok3:>6d}tok | {c5}/{t5} ~{tok5:>7d}tok")
print(f"\n(Appels API sur cette cellule : ~{len(_pbs_bon) * 9})")

Best-of-N sur le modele cheap (zone non-saturee p4-p6) :

Probleme         |            N=1 |                N=3 |                  N=5
--------------------------------------------------------------------------


p4_palindrome    | 3/4 ~   96tok | 4/4 ~   229tok | 4/4 ~    350tok


[(), (1,), (1, 2), (1, 2, 3), (1, 3), (2,), (2, 3), (3,)]


[(), (1,), (1, 2), (1, 2, 3), (1, 3), (2,), (2, 3), (3,)]


[(), (1,), (1, 2), (1, 2, 3), (1, 3), (2,), (2, 3), (3,)]


...

[(), (1,), (1, 2), (1, 2, 3), (1, 3), (2,), (2, 3), (3,)]


[(), (1,), (1, 2), (1, 2, 3), (1, 3), (2,), (2, 3), (3,)]


[(), (1,), (1, 2), (1, 2, 3), (1, 3), (2,), (2, 3), (3,)]
p5_subsets       | 3/3 ~   23tok | 3/3 ~    84tok | 3/3 ~    134tok


[['A', 'C', 'F'], ['A', 'B', 'E', 'F']]


p6_graph         | 2/3 ~   40tok | 2/3 ~   148tok | 2/3 ~    200tok

(Appels API sur cette cellule : ~27)


### Interpretation : Best-of-N ne casse pas l'erreur systematique

**Resultats mesures** (cheap / Llama-3.3-70b) :

| Probleme | N=1 | N=3 | N=5 |
|----------|-----|-----|-----|
| p4 palindrome | 3/4 (~96tok) | 4/4 (~229tok) | 4/4 (~350tok, sature) |
| p5 subsets | 3/3 (~23tok) | 3/3 (~84tok) | 3/3 (~134tok) |
| p6 graph | 2/3 (~40tok) | 2/3 (~148tok) | 2/3 (~200tok) |

**Insight cible** : sur p6, le Best-of-N passe de 2/3 a... 2/3, peu importe N. Pourquoi ? Parce que l'erreur est **systematique** : le modele rate *toujours* le meme test (le tri par longueur puis lexicographique), de la meme maniere. Voter sur des tirages qui se trompent tous pareil ne change rien.

> C'est la **lecon centrale** : la valeur du test-time scaling depend de la **structure d'erreur**. Erreur aleatoire -> BoN aide (comme sur p4, ou l'on passe de 3/4 a 4/4). Erreur systematique -> il faut une technique qui **corrige** (Reflexion), pas qui **vote**.

## Moteur 2 - Reflexion (generateur -> critique -> memoire)

**Self-Refine** (Madaan et al., 2023) et **Reflexion** (Shinn et al., 2023) : une boucle ou le modele genere, puis **se critique** a la lumiere d'un signal externe, puis se regenere. Le signal externe est ici l'**execution des tests** - c'est l'element qui casse les erreurs systematiques.

C'est le mode **"Contextual"** d'ICR : 3 agents (generateur, critiqueur, memoire) en boucle. Contrairement au Best-of-N qui vote, la Reflexion **utilise le feedback** pour corriger la cause de l'erreur.

**Hypothese** : sur p6 (erreur systematique), la Reflexion devrait reussir la ou BoN echouait.

In [9]:
def reflexion_code(probleme, model=FAST_MODEL, iterations=3, max_tokens=2000):
    """Boucle generateur -> execution -> critique -> regeneration.
    A chaque iteration : on genere le code, on l'execute, et si des tests
    echouent on demande au modele de corriger en lui montrant lesquels.
    Le "memoire" = la trace des tests echoues.
    Renvoie (meilleur_passes, total, total_tokens)."""
    total = len(probleme["tests"])
    meilleur_passes = 0
    total_tokens = 0
    feedback = ""  # memoire : ce qui a echoue
    code_courant = ""

    for _it in range(iterations):
        # 1. Generation (ou regeneration si on a un feedback)
        if feedback:
            prompt = (
                probleme["spec"]
                + "\n\nVoici ton code precedent qui ECCHOUE sur certains tests :\n```python\n"
                + code_courant + "\n```\n"
                + "Tests qui echouent :\n" + feedback
                + "\nCorrige le code pour passer TOUS les tests. "
                "Reponds UNIQUEMENT avec le code Python dans un bloc ```python```."
            )
        else:
            prompt = (
                probleme["spec"]
                + "\n\nReponds UNIQUEMENT avec le code Python dans un bloc ```python```."
            )
        resp = chat(prompt, model=model, temperature=0.0, max_tokens=max_tokens)
        total_tokens += len(resp.split())
        code_courant = extraire_code(resp)

        # 2. Execution + diagnostic
        passes, _ = executer_tests(code_courant, probleme)
        if passes > meilleur_passes:
            meilleur_passes = passes
        if passes == total:
            break  # tout passe, on arrete

        # 3. Construction du feedback (memoire) : quels tests echouent ?
        diagnostics = []
        ns = {}
        try:
            exec(code_courant, ns)
        except Exception as exc:
            diagnostics.append(f"ERREUR d'execution : {exc}")
        else:
            for cond in probleme["tests"]:
                try:
                    if not eval(cond, ns):
                        diagnostics.append(f"  ECHEC : {cond}")
                except Exception as exc:
                    diagnostics.append(f"  EXCEPTION sur '{cond}' : {exc}")
        feedback = "\n".join(diagnostics)

    return meilleur_passes, total, total_tokens


print("reflexion_code defini.")

reflexion_code defini.


In [10]:
# Reflexion sur p6 (la ou BoN a echoue : erreur systematique)
_pbs_reflex = [p for p in PROBLEMES if p["diff"] == 6]
if BATCH_MODE:
    _pbs_reflex = _pbs_reflex[:1]

print("Reflexion sur le modele cheap (p6, la ou BoN echouait) :\n")
print(f"{'Probleme':16s} | {'Baseline':>16s} | {'BoN=5':>16s} | {'Reflexion':>22s}")
print("-" * 76)
for pb in _pbs_reflex:
    cb, tb, tokb, _ = best_of_n_code(pb, model=FAST_MODEL, n=1)
    c5, t5, tok5, _ = best_of_n_code(pb, model=FAST_MODEL, n=5)
    cr, tr, tokr = reflexion_code(pb, model=FAST_MODEL, iterations=3)
    print(f"{pb['id']:16s} | {cb}/{tb} ~{tokb:>5d}tok | {c5}/{t5} ~{tok5:>5d}tok | {cr}/{tr} ~{tokr:>8d}tok")
print("\n(Appels API : ~3 a 9 pour la Reflexion)")

Reflexion sur le modele cheap (p6, la ou BoN echouait) :

Probleme         |         Baseline |            BoN=5 |              Reflexion
----------------------------------------------------------------------------


[['A', 'C', 'F'], ['A', 'B', 'E', 'F']]


p6_graph         | 2/3 ~   40tok | 2/3 ~  244tok | 2/3 ~     475tok

(Appels API : ~3 a 9 pour la Reflexion)


### Interpretation : Reflexion, et ses limites

La ou le Best-of-N restait bloque a 2/3 (erreur systematique), la Reflexion dispose theoriquement de l'**information supplementaire** decisive : *quel test echoue, et pourquoi*. En remontrant au modele la trace d'execution ("ECHEC : le tri n'est pas par longueur puis lexicographique"), on lui donne de quoi corriger la **cause** de l'erreur plutot que de la rejouer.

**Resultat mesure** : sur p6, la Reflexion atteint elle aussi **2/3** (~475tok) -- identique au baseline (2/3, ~40tok) et au BoN=5 (2/3, ~244tok). **La Reflexion n'a pas casse l'erreur systematique non plus**, malgré le feedback explicite sur le test qui échoue.

**Pourquoi la Reflexion échoue ici** : le feedback dit *quel* test échoue, mais le modèle cheap (Llama-3.3-70b) n'arrive pas, même avec cette information, à produire le tri correct (par longueur puis lexicographique). Il persiste dans une approche alternative. C'est la limite fondamentale :

> **Lecon centrale** : le test-time scaling a ses **limites**. Aucune technique (BoN, ni meme Reflexion avec feedback) ne corrige une erreur que le modèle ne *comprend* pas. Le gain dépend de la **capacité du modèle à exploiter le signal**, pas seulement de la quantité de calcul dépensée. C'est précisément pourquoi un **routeur adaptatif** (section suivante) reste utile -- non pour garantir le succès, mais pour **escalader intelligemment** et savoir quand s'arrêter.

**Comparaison des strategies selon la structure d'erreur** :

| Type d'erreur | Technique | Efficace ? |
|---------------|-----------|------------|
| Aleatoire | Best-of-N (vote) | Oui -- les tirages s'annulent (cf. p4 : 3/4 -> 4/4) |
| Systematique (corrigible) | Reflexion (feedback) | Oui, *si* le modèle exploite le signal |
| Systematique (non-corrigible) | Aucune | Non -- p6 en est l'exemple (2/3 partout) |
| Aucune (sature) | Aucune | Rien a corriger |

## Moteur 3 - Tree-of-Thoughts (recherche sur etats partiels)

**Tree-of-Thoughts** (Yao et al., 2023) : au lieu de raisonner en ligne droite, on explore un **arbre d'etats intermediaires**, on evalue chaque branche, et on developpe les plus prometteuses (recherche en faisceau / beam search). C'est le mode **"Deepthink"** d'ICR.

**Domaine de predilection** : les problemes **combinatoires** (24-game, mots croises, planification) ou il existe des etats intermediaires evaluables. La generation de code *pure* est un **mauvais fit** pour ToT (pas d'etat partiel naturel a evaluer). Nous le demontrons donc sur le **24-game** - le cas canonique de l'article original.

> Dans l'article ToT original, l'**evaluateur** de branche est un LLM. Ici, pour garder une latence raisonnable, nous utilisons un evaluateur **deterministe** (test de joignabilite de 24). La structure de recherche (arbre + elagage en faisceau) reste identique.

In [11]:
from itertools import combinations


def atteignable_24(nombres, cible=24.0, tol=1e-6):
    """Verifie recursivement (exhaustif sur le petit arbre) si cible est
    atteignable depuis cette liste de nombres. Sert d'evaluateur 'parfait'
    pour le beam search."""
    n = len(nombres)
    if n == 1:
        return abs(nombres[0] - cible) < tol
    for i, j in combinations(range(n), 2):
        a, b = nombres[i], nombres[j]
        rest = [nombres[k] for k in range(n) if k not in (i, j)]
        for val in (a + b, a - b, b - a, a * b):
            if atteignable_24(rest + [val], cible, tol):
                return True
        if abs(b) > tol and atteignable_24(rest + [a / b], cible, tol):
            return True
        if abs(a) > tol and atteignable_24(rest + [b / a], cible, tol):
            return True
    return False


def evaluator(etat):
    """Heuristique pour le beam search : privilegie les etats d'ou 24 est
    atteignable. Dans l'article ToT original, ce role est tenu par un LLM."""
    nombres = etat
    if len(nombres) == 1:
        return 1000.0 - abs(nombres[0] - 24.0)
    return 500.0 if atteignable_24(nombres) else -min(abs(x - 24.0) for x in nombres)


def tot_bfs_24(nombres, largeur=3, profondeur=4):
    """Tree-of-Thoughts (BFS avec faisceau) sur le 24-game.
    Parametres petits pour la latence. Renvoie la liste des expressions
    (chaines) trouvees qui valent 24."""
    etats = [(list(map(float, nombres)), [str(x) for x in nombres])]
    solutions = []
    for _ in range(profondeur):
        candidats = []
        for valeurs, exprs in etats:
            if len(valeurs) < 2:
                if len(valeurs) == 1 and abs(valeurs[0] - 24.0) < 1e-6:
                    solutions.append(exprs[0])
                continue
            for i, j in combinations(range(len(valeurs)), 2):
                a, b = valeurs[i], valeurs[j]
                ea, eb = exprs[i], exprs[j]
                rest_v = [valeurs[k] for k in range(len(valeurs)) if k not in (i, j)]
                rest_e = [exprs[k] for k in range(len(exprs)) if k not in (i, j)]
                for txt, val in (
                    (f"({ea}+{eb})", a + b),
                    (f"({ea}-{eb})", a - b),
                    (f"({eb}-{ea})", b - a),
                    (f"({ea}*{eb})", a * b),
                ):
                    candidats.append((rest_v + [val], rest_e + [txt]))
                if abs(b) > 1e-9:
                    candidats.append((rest_v + [a / b], rest_e + [f"({ea}/{eb})"]))
                if abs(a) > 1e-9:
                    candidats.append((rest_v + [b / a], rest_e + [f"({eb}/{ea})"]))
        # faisceau : on garde les 'largeur' meilleurs etats
        candidats.sort(key=lambda c: evaluator(c[0]), reverse=True)
        etats = candidats[:largeur]
        for valeurs, exprs in etats:
            if len(valeurs) == 1 and abs(valeurs[0] - 24.0) < 1e-6:
                if exprs[0] not in solutions:
                    solutions.append(exprs[0])
    return solutions


# Demonstration sur 2 instances canoniques du 24-game
for _nb in ([4, 1, 8, 7], [8, 3, 8, 3]):
    sols = tot_bfs_24(_nb, largeur=3, profondeur=4)
    print(f"24-game {_nb} -> {len(sols)} solution(s) : {sols[:3]}")
print("\n(Aucun appel API : evaluateur deterministe pour garder une latence faible)")

24-game [4, 1, 8, 7] -> 6 solution(s) : ['(8*(7-(4*1)))', '(8*(7-(4/1)))', '((4-8)*(1-7))']
24-game [8, 3, 8, 3] -> 4 solution(s) : ['(8/(3-(8/3)))', '(8/(3-(8/3)))', '(8/(3-(8/3)))']

(Aucun appel API : evaluateur deterministe pour garder une latence faible)


## Moteur 4 - Routeur adaptatif (ICR "Adaptive Deepthink")

Aucune technique n'est universelle. Le routeur **estime la difficulte** d'un probleme (1 appel), puis choisit la strategie au cout croissant :

1. **Baseline** (1 appel) - si tout passe, on s'arrete (tache facile/saturee).
2. **Best-of-N** (N appels) - si erreur aleatoire suspectee.
3. **Reflexion** (feedback) - si erreur systematique.

C'est le principe d'**ICR** : ne pas depenser du calcul la ou ce n'est pas necessaire, et escalader intelligemment sur les taches qui le meritent.

In [12]:
def estimer_difficulte(probleme, model=FAST_MODEL, max_tokens=200):
    """Demande au LLM d'estimer la difficulte (1-6). 1 appel."""
    prompt = (
        "Voici un probleme de programmation Python :\n"
        + probleme["spec"]
        + "\n\nEstime sa difficulte algorithmique sur une echelle de 1 (trivial) a 6 "
        "(algorithmique avance). Reponds UNIQUEMENT par un entier entre 1 et 6."
    )
    resp = chat(prompt, model=model, temperature=0.0, max_tokens=max_tokens)
    try:
        diff = int(resp.strip())
        return max(1, min(6, diff))
    except Exception:
        return 3  # valeur neutre par defaut


def routeur_adaptatif(probleme, model=FAST_MODEL):
    """Estime la difficulte, puis choisit une strategie au cout croissant.
    Renvoie un dict {id, diff_estimee, strategie, passes, total, tokens, appels}."""
    diff_est = estimer_difficulte(probleme, model=model)
    total = len(probleme["tests"])

    # 1. Toujours essayer la baseline d'abord (cheap)
    code_gen, tok1 = gen_code(probleme, model=model, temperature=0.0)
    passes, _ = executer_tests(code_gen, probleme)
    tokens = tok1
    appels = 2  # estimation + baseline
    strategie = "baseline"

    # 2. Si baseline echoue et difficulte moyenne -> Best-of-N=3
    if passes < total and diff_est >= 4:
        c3, _, tok3, _ = best_of_n_code(probleme, model=model, n=3)
        tokens += tok3
        appels += 3
        if c3 > passes:
            passes = c3
        strategie = "best_of_n(3)"

    # 3. Si BoN insuffisant -> Reflexion (feedback systematique)
    if passes < total:
        cr, _, tokr = reflexion_code(probleme, model=model, iterations=2)
        tokens += tokr
        appels += 2
        if cr > passes:
            passes = cr
        strategie = "reflexion"

    return {
        "id": probleme["id"], "diff_estimee": diff_est,
        "strategie": strategie, "passes": passes, "total": total,
        "tokens": tokens, "appels": appels,
    }


# Test sur 3 problemes representatifs (facile, moyen, dur)
_pbs_route = [PROBLEMES[0], PROBLEMES[3], PROBLEMES[5]]
if BATCH_MODE:
    _pbs_route = _pbs_route[:2]
print("Routeur adaptatif :\n")
print(f"{'Probleme':16s} | {'Diff est.':>9s} | {'Strategie':>14s} | {'Resultat':>12s} | {'Tokens':>7s} | {'Appels':>6s}")
print("-" * 72)
for pb in _pbs_route:
    r = routeur_adaptatif(pb, model=FAST_MODEL)
    print(f"{r['id']:16s} | {r['diff_estimee']:>9d} | {r['strategie']:>14s} | {r['passes']}/{r['total']:<3d}      | {r['tokens']:>7d} | {r['appels']:>6d}")
print("\n(Appels API : ~6 a 10)")

Routeur adaptatif :

Probleme         | Diff est. |      Strategie |     Resultat |  Tokens | Appels
------------------------------------------------------------------------


p1_carre         |         1 |       baseline | 3/3        |       8 |      2


p4_palindrome    |         5 |       baseline | 4/4        |      70 |      2


p6_graph         |         4 |      reflexion | 2/3        |     427 |      7

(Appels API : ~6 a 10)


## Synthese : size-scaling vs test-time scaling

Le tableau ci-dessous resume les resultats mesures et incarne la **these centrale**.

| Strategie | Probleme | Resultat | Tokens | Lecture |
|-----------|----------|----------|--------|---------|
| Single-shot cheap | p4 | 4/4 | ~70 | Resout, economique |
| Single-shot big | p4 | **0/4** | ~0 | Sortie vide : big echoue la ou cheap passe |
| Single-shot cheap | p5 | 3/3 | ~23 | Resout, economique |
| Single-shot big | p5 | 3/3 | ~33 | Resout, legerement plus cher |
| Single-shot cheap | p6 | **2/3** | ~40 | Erreur systematique |
| Single-shot big | p6 | **0/3** | ~0 | Sortie vide : big echoue la ou cheap reussit partiellement |
| BoN=5 cheap | p6 | 2/3 | ~200 | Le vote ne casse pas l'erreur systematique |
| Reflexion cheap | p6 | 2/3 | ~475 | Le feedback non plus : limite du test-time scaling |
| Routeur cheap | p6 | 2/3 | ~427 | Escalade (baseline -> BoN -> reflexion) sans casser l'erreur |

**La these en 3 points** :

1. **Le size-scaling n'est pas universellement superieur.** Sur p4 et p6, le modele gros raisonnant *echoue* (sortie vide) la ou le modele bon marche reussit. Le raisonnement peut etre un piege -- voire conduire a un echec total.
2. **Le test-time scaling n'est pas un supplement universel.** Sur p1-p3 (sature), aucune strategie n'ajoute rien. Depenser du calcul la est du gaspillage. Et sur p6, *aucune* technique test-time (BoN, Reflexion, routeur) ne casse l'erreur systematique.
3. **Le gain depend de la structure d'erreur -- et de la capacite du modele.** Erreur aleatoire -> Best-of-N aide (p4 : 3/4 -> 4/4). Erreur systematique non-corrigible -> aucune technique n'aide (p6 : 2/3 partout).

> **Conclusion** : optimiser un LLM, c'est naviguer un **espace 2D** (taille du modele x calcul a l'inference). Les deux axes **ne sont pas equivalents** et ne se substituent pas librement. Un modele bon marche bien orchestre peut battre un modele gros pour une fraction du cout - *sur certaines taches*. Mais le test-time scaling n'est pas magique : il est inutile sur les taches saturees et impuissant face aux erreurs que le modele ne comprend pas.

## Exercice 1 - Best-of-N pondere

Le Best-of-N classique vote a egalite. Mais toutes les solutions ne se valent pas : celles qui passent **plus de tests** meritent plus de poids.

**Objectif** : implementer `best_of_n_pondere(probleme, model, n)` qui genere N solutions et renvoie celle qui passe le **maximum de tests** (et, a egalite, la premiere trouvee).

**Indice** : reutilisez `gen_code` et `executer_tests`. La difference avec `best_of_n_code` : renvoyez aussi le *code* de la meilleure solution, pas seulement le score.

In [13]:
def best_of_n_pondere(probleme, model=FAST_MODEL, n=5):
    """Best-of-N qui renvoie la solution passant le plus de tests.
    Renvoie (meilleur_code, meilleur_passes, total, tokens).
    TODO etudiant : completez la boucle."""
    meilleur_code = ""
    meilleur_passes = 0
    total = len(probleme["tests"])
    tokens = 0
    # TODO etudiant : generer n solutions, executer chacune, garder la meilleure
    # Etape 1 : boucle for i in range(n)
    # Etape 2 : code_gen, tok = gen_code(...)
    # Etape 3 : passes, _ = executer_tests(...)
    # Etape 4 : si passes > meilleur_passes : memoriser le code et le score
    result = None  # TODO etudiant
    return meilleur_code, meilleur_passes, total, tokens


# Test (renvoie des valeurs nulles tant que l'exercice n'est pas complete)
_code, _p, _t, _tok = best_of_n_pondere(PROBLEMES[2], model=FAST_MODEL, n=3)
print(f"Exercice 1 - resultat actuel : {_p}/{_t} tests, ~{_tok} tokens (a completer)")

Exercice 1 - resultat actuel : 0/3 tests, ~0 tokens (a completer)


## Exercice 2 - Diversite dans le pool BoN

Si les N solutions generees sont **trop similaires**, le vote perd de son interet. On penalise les solutions proches de celles deja selectionnees.

**Objectif** : implementer `best_of_n_divers(probleme, model, n)` qui selectionne des solutions diversifiees. On fournit `similarite(a, b)` (recouvrement de mots entre deux codes).

**Indice** : pour chaque candidat, calculez sa similarite max avec les deja-selectionnes ; ne le gardez que si cette similarite est inferieure au seuil.

In [14]:
def similarite(a, b):
    """Recouvrement de mots entre deux codes (0 a 1). Deja fourni."""
    mots_a = set(a.split())
    mots_b = set(b.split())
    if not mots_a and not mots_b:
        return 1.0
    return len(mots_a & mots_b) / max(1, len(mots_a | mots_b))


def best_of_n_divers(probleme, model=FAST_MODEL, n=5, seuil=0.8):
    """Best-of-N avec penalite de similarite.
    Renvoie (codes_selectionnes, meilleur_passes, total, tokens).
    TODO etudiant : completez la selection diversifiee."""
    codes_selectionnes = []
    total = len(probleme["tests"])
    tokens = 0
    meilleur_passes = 0
    # TODO etudiant : generer n solutions, ne garder que celles assez diverses
    # Etape 1 : for i in range(n) -> code_gen, tok = gen_code(...)
    # Etape 2 : sim_max = max((similarite(code_gen, sel) for sel in codes_selectionnes), default=0.0)
    # Etape 3 : si sim_max < seuil : ajouter a codes_selectionnes
    # Etape 4 : suivre meilleur_passes parmi les selectionnes via executer_tests
    result = None  # TODO etudiant
    return codes_selectionnes, meilleur_passes, total, tokens


_codes, _p, _t, _tok = best_of_n_divers(PROBLEMES[2], model=FAST_MODEL, n=3)
print(f"Exercice 2 - {len(_codes)} solution(s) selectionnee(s), meilleur {_p}/{_t} (a completer)")

Exercice 2 - 0 solution(s) selectionnee(s), meilleur 0/3 (a completer)


## Exercice 3 - Double verification (code + tests generes)

Jusqu'ici les tests sont **fixes** (fournis par le benchmark). Une approche plus robuste : faire generer au LLM **a la fois** le code ET une suite de tests supplementaires, puis croiser les deux.

**Objectif** : implementer `reflexion_double_verif(probleme, model, iterations)` qui, a chaque tour, genere le code **et** une fonction de test, execute les deux, et s'en sert comme feedback croise.

**Indice** : demandez au modele deux blocs (un pour le code, un pour une fonction `tests_supplementaires()` renvoyant une liste de booleens). C'est l'exercice le plus avance : commencez par generer seulement la fonction de test supplementaire et l'executer.

In [15]:
def reflexion_double_verif(probleme, model=FAST_MODEL, iterations=2):
    """Reflexion avec code + tests generes croises.
    Renvoie (meilleur_passes, total, tokens).
    TODO etudiant : c'est l'exercice le plus avance."""
    total = len(probleme["tests"])
    meilleur_passes = 0
    tokens = 0
    # TODO etudiant : a chaque iteration
    # Etape 1 : prompt = specifier code + fonction tests_supplementaires()
    # Etape 2 : extraire les deux blocs de code (via extraire_code)
    # Etape 3 : executer code + tests fournis + tests supplementaires
    # Etape 4 : si echec, feedback croise et regenerer
    result = None  # TODO etudiant
    return meilleur_passes, total, tokens


_mp, _mt, _mtok = reflexion_double_verif(PROBLEMES[5], model=FAST_MODEL, iterations=2)
print(f"Exercice 3 - double verification : {_mp}/{_mt} tests, ~{_mtok} tokens (a completer)")

Exercice 3 - double verification : 0/3 tests, ~0 tokens (a completer)


## Conclusion

Nous avons reconstruit les **4 moteurs** de test-time scaling et mesure leurs compromis :

| Moteur | Principe | Budget | Quand l'utiliser |
|--------|----------|--------|------------------|
| Best-of-N | Vote sur N tirages | N appels | Erreur aleatoire |
| Reflexion | Generateur + critique (feedback) | 2 a 3x appels | Erreur systematique |
| Tree-of-Thoughts | Recherche sur etats partiels | variable | Probleme combinatoire |
| Routeur adaptatif | Estime difficulte, escalade | adaptatif | Quand on ne sait pas a l'avance |

**These retenue** : optimiser un LLM, c'est un **compromis 2D** - taille du modele (size-scaling) x calcul a l'inference (test-time scaling). Ces axes **ne sont pas equivalents** : un modele bon marche bien orchestre peut battre un modele gros raisonnant, pour moins de tokens, *sur les bonnes taches*. Mais le test-time scaling n'est pas magique - il est inutile sur les taches saturees et impuissant face a certaines erreurs profondes.

**Vers le projet ICR** : les 4 moteurs ci-dessus correspondent aux modes d'ICR (https://github.com/ryoiki-tokuiten/Iterative-Contextual-Refinements) : Contextual (Reflexion), Deepthink (ToT), Adaptive (routeur). La suite logique : les **courbes de scaling** de Snell (test-time compute optimal), la comparaison avec les modeles raisonnants natifs, et l'encapsulation en plugin Semantic Kernel.

**Prochaines etapes** (voir issue #2926) :
- Courbes de scaling test-time (Snell 2024) ;
- Test-time scaling vs modeles raisonnants natifs (gpt-5-nano) ;
- Encapsulation des moteurs en plugin Semantic Kernel.

## References

- **Snell, Charikar, Xu (2024)** - *Scaling LLM Test-Time Compute Optimally*. Formalise le second axe de scaling (calcul a l'inference).
- **Wang et al. (2022)** - *Self-Consistency Improves Chain of Thought Reasoning*. Best-of-N par vote.
- **Madaan et al. (2023)** - *Self-Refine: Iterative Refinement with Self-Feedback*. Generateur -> critique.
- **Shinn et al. (2023)** - *Reflexion: Language Agents with Verbal Reinforcement Learning*. Boucle avec memoire.
- **Yao et al. (2023)** - *Tree of Thoughts: Deliberate Problem Solving with Large Language Models*. Recherche sur etats.
- **Projet ICR** - https://github.com/ryoiki-tokuiten/Iterative-Contextual-Refinements (modes Contextual, Deepthink, Adaptive).

---

**Navigation** : [Index](README.md) | [<< Precedent](11_Quantization.ipynb)